# 01 · The book, the splits, and the labels

**pragmatiq series, notebook 1 of 4 after the [tour](00_blueprint_v2_tour.ipynb).** Over four notebooks you take
apart a pretrained transaction foundation model (TFM): a network pretrained on
many users' event histories to produce one embedding per user, which
downstream tasks such as fraud and churn then reuse. Each notebook covers one
or two *slots* of the kit's building-block taxonomy:

| Slot | Question it answers | Blueprint v2 choice | Notebook |
|---|---|---|---|
| 0 Data | Which events, labels, and cutoffs? | pragmatiq book, temporal split, spliced identity theft | 01 |
| 1 Event representation | How does one event become tokens? | key-value tokens, hashed identifiers, TimeRoPE | 02 |
| 2 Sequence architecture | How do events become a history state? | profile encoder + event encoder + causal decoder | 03b |
| 3 Objective | What is pretraining asked to predict? | the next event's fields + within-event masking | 03b |
| 4 Readout and adaptation | Which state is the embedding, and how is it used? | last token; LoRA fine-tuning per task | 04 |

**This notebook covers slot 0, the data.** By the end you will know the
shape of the book, the three time cutoffs that keep the future out of every
input, and the five downstream tasks the series scores.

## Before you start

| | |
|---|---|
| Hardware | one NVIDIA GPU and about 30 GB of host memory for the default 4,000-user sample (about 7.7 GB per 1,000 users) |
| Time | about 4 minutes for this notebook, most of it spent preparing the sample |
| Downloads | the book (1.8 GB) and the pretrained checkpoint (45 MB), once, into `~/.cache/tfm/lab` |

`fetch_lab_assets` finds both. It uses `TFM_PRAGMATIQ_ROOT` (the book) and
`TFM_V2_CHECKPOINT` (a checkpoint directory) when they point at local copies,
then an earlier download, and downloads them otherwise. To use your own
checkpoint, point `TFM_V2_CHECKPOINT` at a directory built by
`scripts/package_lab_assets.py` (notebook 05).

> **TODO (hosting):** the download URLs in `tfm.runtime.lab` are placeholders
> until the archives are hosted. Until then, set `TFM_PRAGMATIQ_ROOT` to a
> local copy of the book and `TFM_V2_CHECKPOINT` to a checkpoint directory
> (the lab's `blueprint_v2_lab_tuned`, or one built by
> `scripts/package_lab_assets.py`).

> **Smoke mode.** CI runs this notebook with `TFM_NOTEBOOK_SMOKE=1`: a
> generated 120-user book, an untrained model, and CPU only. Every cell runs,
> but the numbers and plots are meaningless. The first code cell prints which
> mode you are in.

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tfm.runtime.lab import fetch_lab_assets, prepare_lab_data
from tfm.training.reproducibility import seed_everything

GREEN, DARK, GREY = "#76B900", "#1A1A1A", "#8C8C8C"
PALETTE = [GREEN, "#0074DF", "#F2A900", "#5D1682", "#E52020", DARK, GREY, "#00A3A3"]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
})

assets = fetch_lab_assets("01")
config = assets.config
seed_everything(config.seed)
print(assets.describe())

## The book

The series reads one dataset, a [pragmatiq](https://github.com/dynamiq-ai/pragmatiq)
Generator v2 *book* (`synth_100k_gen2_m31_seed0`): 100,000 synthetic users
over 31 months. Each user has a profile and one event stream that mixes
several *sources*: card and bank transactions, app sessions, marketing
messages, and trading. The book ships labels for several downstream
tasks on the same users, which is what lets one pretrained model be tested on
several tasks. It is generated, so no real customer is in it.

`prepare_lab_data` runs the whole data slot on a random sample of users:

1. load the book and validate every event against the `TransactionEvent` contract;
2. split every history by time into train, validation, and test periods;
3. tokenize with the checkpoint's own tokenizer (notebook 02);
4. build one sequence per user at each cutoff, and the rows of each task.

On the lab GPU this takes about two minutes for the default sample of
4,000 users (`LAB_USERS`); `fetch_lab_assets(..., n_users=...)` takes more.

In [ ]:
data = prepare_lab_data(assets)
print(f"{len(data.events):,} events from {len(data.profiles):,} users")
print("tasks labelled in the book:", ", ".join(sorted(data.label_rows)))

### One event per source

Every event has a user, a UTC timestamp, and a free-form set of fields. A
card payment and an app tap carry different fields, and the kit keeps them
in one stream, recording the source of each.

In [ ]:
examples = {}
for event in data.events[:200_000]:
    examples.setdefault(event.categorical.get("source", "unknown"), event)
for source, event in examples.items():
    fields = {**event.categorical, **{k: round(v, 2) for k, v in event.numeric.items()}}
    fields.pop("source", None)
    print(f"{source:<12} {event.event_timestamp:%Y-%m-%d %H:%M}  {fields}")

## Exploring the book

A table with one row per event makes the exploration quick. (The kit itself
never needs pandas; the notebook uses it for plotting.)

In [ ]:
events = pd.DataFrame({
    "user": [e.entity_id for e in data.events],
    "ts": pd.to_datetime([e.event_timestamp for e in data.events], utc=True),
    "source": [e.categorical.get("source", "unknown") for e in data.events],
    "txn_type": [e.categorical.get("txn_type") for e in data.events],
    "amount": [e.numeric.get("amount", np.nan) for e in data.events],
})
counts = events["source"].value_counts()
per_user = events.groupby("user").size()
summary = events.groupby("source").agg(events=("user", "size"), users=("user", "nunique"))
summary.loc[counts.index]

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 3.8))
left.barh(counts.index[::-1], counts.to_numpy()[::-1], color=GREEN)
left.set_xscale("log")
left.set_title("Events per source")
left.set_xlabel("events (log scale)")

bins = np.geomspace(max(per_user.min(), 1), per_user.max() + 1, 40)
right.hist(per_user, bins=bins, color=GREEN, edgecolor="white")
right.set_xscale("log")
right.axvline(config.data.sequence_length, color=DARK, ls="--")
right.annotate(f" the model reads the last {config.data.sequence_length}",
               (config.data.sequence_length, 0.92), xycoords=("data", "axes fraction"))
right.set_title("Events per user, whole book")
right.set_xlabel("events (log scale)")
right.set_ylabel("users")
plt.tight_layout()
plt.show()

Transactions and app events dominate. Most users have far more events than
the 128 the model reads (dashed line). Each sequence keeps the most recent
events before its cutoff, so the model sees the last few weeks of a history.

### Each source has its own fields

The heatmap shows the share of each source's events that carry each field.
A card payment has an `mcc` (merchant category code) and a `merchant`, an app
event has a `screen` and a `device_id`. The key-value tokenizer (notebook 02)
needs no fixed schema, which is why one model can read every source.

In [ ]:
rng = np.random.default_rng(config.seed)
sample = rng.choice(len(data.events), size=min(100_000, len(data.events)), replace=False)
present = pd.DataFrame(
    [dict.fromkeys([*data.events[i].categorical, *data.events[i].numeric], 1.0) for i in sample]
).fillna(0.0)
coverage = present.drop(columns="source").groupby(events["source"].to_numpy()[sample]).mean()
coverage = coverage.reindex([s for s in counts.index if s in coverage.index])
order = sorted(coverage.columns, key=lambda k: (coverage[k].to_numpy().argmax(), -coverage[k].max()))
coverage = coverage[order]

fig, ax = plt.subplots(figsize=(12, 0.45 * len(coverage) + 1.6))
ax.imshow(coverage.to_numpy(), cmap="Greens", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(coverage.shape[1]), coverage.columns, rotation=60, ha="right")
ax.set_yticks(range(len(coverage)), coverage.index)
ax.grid(False)
ax.set_title("Share of each source's events that carry a field")
plt.tight_layout()
plt.show()

### Amounts span four orders of magnitude

Salaries, rent, cash withdrawals, and coffee all share one `amount` field.
Notebook 02 shows how the tokenizer turns this skewed range into 16 buckets
of equal frequency.

In [ ]:
txn = events.dropna(subset=["amount", "txn_type"]).assign(amount=lambda f: f["amount"].abs())
txn = txn[txn["amount"] > 0]
types = txn["txn_type"].value_counts().index[:8]
fig, ax = plt.subplots(figsize=(12, 3.8))
parts = ax.boxplot([txn.loc[txn["txn_type"] == t, "amount"] for t in types], showfliers=False,
                   patch_artist=True, medianprops={"color": DARK})
for box in parts["boxes"]:
    box.set(facecolor=GREEN, alpha=0.6)
ax.set_xticks(range(1, len(types) + 1), types, rotation=20)
ax.set_yscale("log")
ax.set_ylabel("|amount| (log scale)")
ax.set_title("Transaction amounts by type (boxes: quartiles; outliers hidden)")
plt.tight_layout()
plt.show()

## Three cutoffs, three label windows

The book is split by time, never at random. `data.temporal_split` sets two
dates, `train_end` and `validation_end`, and the book's last event closes the
test period. A sequence built for a side holds only events at or before that
side's cutoff.

Labels get windows of their own. A side's fraud label counts only fraud
inside that side's window. Windows are half-open, `(start, end]`, so an event
on a boundary belongs to exactly one side. Earlier events stay in the
history as inputs, but their labels do not count.

The forecast tasks come with the book. At 2025-01-01, `churn_6m` asks
whether the user goes quiet in the next six months, and `ltv_positive`
whether those six months are profitable. At 2024-07-01, `default_12m` asks
whether the user defaults within a year. All three are issued after the
train cutoff, so the pretrained encoder never saw the months they predict.

In [ ]:
for side, window in data.cutoffs.windows.items():
    print(f"{side:>10} label window: ({window.start:%Y-%m-%d}, {window.end:%Y-%m-%d}]")

In [ ]:
events["month"] = events["ts"].dt.tz_convert(None).dt.to_period("M").dt.to_timestamp()
monthly = events.groupby(["month", "source"]).size().unstack(fill_value=0)[counts.index]

fig, ax = plt.subplots(figsize=(12, 4.2))
ax.stackplot(monthly.index, monthly.T.to_numpy(), labels=monthly.columns, alpha=0.85)
first, last = monthly.index.min(), monthly.index.max() + pd.offsets.MonthEnd(1)
shade = {"train": GREEN, "validation": "#F2A900", "test": "#0074DF"}
for side, window in data.cutoffs.windows.items():
    start = max(pd.Timestamp(window.start).tz_convert(None), first)
    ax.axvspan(start, pd.Timestamp(window.end).tz_convert(None), color=shade[side], alpha=0.08)
    ax.annotate(f" {side}", (start, 0.02), xycoords=("data", "axes fraction"), color=shade[side],
                fontweight="bold")
issued = {}
for task, rows in data.label_rows.items():
    if task not in ("churn_6m", "ltv_positive", "default_12m"):
        continue
    issued.setdefault(pd.Timestamp(rows[0].timestamp).tz_convert(None), []).append(task)
for when, tasks in issued.items():
    ax.axvline(when, color=DARK, ls=":", lw=1)
    ax.annotate(" " + ", ".join(sorted(tasks)), (when, 0.97), xycoords=("data", "axes fraction"),
                fontsize=8, va="top")
ax.set_xlim(first, last)
ax.set_ylabel("events per month")
ax.set_title("Monthly events, the three label windows (shaded), and when each forecast is issued")
ax.legend(loc="upper left", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

### One user's recent history

What the model reads for one user: an irregular stream from several sources,
with bursts of app activity between card payments and the occasional
marketing message. Marker area follows the amount. The user is a typical one:
median activity, and every source present.

In [ ]:
import matplotlib.dates as mdates

cutoff = pd.Timestamp(data.cutoffs.train)
before = events[(events["ts"] <= cutoff) & (events["ts"] > cutoff - pd.Timedelta(days=21))]
variety = before.groupby("user")["source"].nunique()
typical = per_user[per_user.between(per_user.quantile(0.4), per_user.quantile(0.6))]
user = variety[variety.index.isin(typical.index)].sort_values(kind="stable").index[-1]
recent = before[before["user"] == user]
shown = [s for s in counts.index if s in set(recent["source"])]

fig, ax = plt.subplots(figsize=(12, 0.5 * len(shown) + 1.6))
for row, source in enumerate(shown):
    part = recent[recent["source"] == source]
    area = 12 + 2 * np.sqrt(part["amount"].abs().fillna(0.0))
    ax.scatter(part["ts"], np.full(len(part), row), s=area, alpha=0.6, color=PALETTE[row % len(PALETTE)])
locator = mdates.AutoDateLocator()
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
ax.set_yticks(range(len(shown)), shown)
ax.set_ylim(-0.7, len(shown) - 0.3)
ax.set_title(f"User {user}: the 21 days before the train cutoff")
plt.tight_layout()
plt.show()

## The fraud task: spliced identity theft

pragmatiq labels fraud too, but one event gives it away: every labelled fraud
is an online payment abroad, and a classifier on that transaction alone
scores 1.000 ROC-AUC. A history adds nothing to that. The kit instead builds
an event-level task from the unmodified book, following PersonaLedger's
identity-theft method:

1. Fix the task's user split first, before any label exists.
2. Make 20% of each side's eligible users *victims*. Give each one a *donor*:
   another user from the same country whose median card amount is within 1.5×
   of the victim's.
3. Pick a theft day in the last 28 days of the side's label window. Copy the
   donor's card transactions and app sessions from that day into the victim's
   history, at their original timestamps. The copied transactions are
   **fraud**.
4. Label the victim's own card transactions in the 7 days before, and on the
   theft day, **legitimate**. Nothing else is labelled.

Each copied transaction is an ordinary one for somebody, so a detector has to
notice that it does not fit *this* user's history. Pretraining and the
forecast tasks read the unspliced book; only the fraud tasks see the splice.

In [ ]:
splice = data.fraud.manifest
print(f"{splice['n_eligible']:,} eligible users; victims per side: {splice['n_victims']}")
print(f"{splice['n_fraud_events']:,} copied fraud events, "
      f"{splice['n_legitimate_events']:,} labelled legitimate events")

record = max((r for r in splice["records"] if r["side"] == "evaluation"),
             key=lambda r: (r["n_fraud"], r["n_legitimate"]))
theft = pd.Timestamp(record["theft_day"], tz="UTC")
spliced = [e for e in data.fraud.events if e.entity_id == record["victim"]
           and theft - pd.Timedelta(days=10) <= e.event_timestamp < theft + pd.Timedelta(days=2)]
print(f"victim {record['victim']}, donor {record['donor']}, theft day {record['theft_day']}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.6))
groups = {
    "victim's own event, unlabelled (app events have no amount)": ([e for e in spliced if "fraud" not in e.labels], GREY, "o"),
    "victim's own card payment (legitimate)": (
        [e for e in spliced if "fraud" in e.labels and e.labels["fraud"].value == 0], "#0074DF", "o"),
    "copied from the donor (fraud)": (
        [e for e in spliced if "fraud" in e.labels and e.labels["fraud"].value == 1], "#E52020", "X"),
}
for label, (part, colour, marker) in groups.items():
    amounts = [abs(e.numeric.get("amount", 0.0)) + 1.0 for e in part]
    ax.scatter([e.event_timestamp for e in part], amounts, color=colour, marker=marker,
               s=45, alpha=0.8, label=f"{label}: {len(part)}")
ax.axvspan(theft, theft + pd.Timedelta(days=1), color="#E52020", alpha=0.08)
ax.set_yscale("log")
ax.set_ylabel("|amount| + 1 (log scale)")
ax.set_title("One evaluation-side victim around the theft day (shaded)")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

The copied payments are in the victim's currency, in the victim's range of
amounts, and at plausible hours. What gives them away is subtler: merchants
this user never visits, a new device in the app session, a different daily
rhythm. That is the kind of evidence a model of the whole history can learn,
and a model of one event cannot.

## The five tasks the series scores

| Task | Level | One row is… | Label |
|---|---|---|---|
| `fraud_user` | user | a user's history at the end of a side's label window | 1 if a theft day was spliced into that window |
| `fraud_txn` | event | one labelled card transaction, with the history up to and including it | 1 if that transaction was copied |
| `churn_6m` | user | a user's history at 2025-01-01 | 1 if the user goes quiet in the next six months |
| `ltv_positive` | user | a user's history at 2025-01-01 | 1 if the next six months are profitable |
| `default_12m` | user | a user's history at 2024-07-01 | 1 if the user defaults within a year |

Both fraud tasks read the same splice. `fraud_user` asks it of the user:
does this history hold a stolen-card day? `fraud_txn` asks it of each
transaction, as Blueprint v1 scored fraud: the last event of each row is the
transaction being judged, and the rows are the victims' labelled
transactions only. Either way the positive share is set by the splice, not
by how rare fraud is. The cell below also prints the copied events' share of
the whole sample, which is far smaller. `fraud_task` (`tfm.runtime.evaluate`)
builds `fraud_user`; `lab_tasks` builds the other four and refuses, with the
reason, any task the sample cannot support.

In [ ]:
from tfm.runtime.evaluate import fraud_entity_split, fraud_task
from tfm.runtime.lab import lab_tasks

tasks = {"fraud_user": fraud_task(data, fraud_entity_split(data, config.seed)),
         **lab_tasks(assets, data, ("fraud_txn", "churn_6m", "default_12m", "ltv_positive"))}
support = pd.DataFrame({name: task.summary() for name, task in tasks.items()}).T
support["train positive share"] = (support["n_train_positive"] / support["n_train"]).round(3)
print(f"copied events in the whole sampled book: {splice['n_fraud_events']:,} of {len(data.events):,} "
      f"({splice['n_fraud_events'] / len(data.events):.3%})")
support

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))
x = np.arange(len(support))
for offset, side, alpha in ((-0.2, "train", 1.0), (0.2, "eval", 0.45)):
    positives = support[f"n_{side}_positive"]
    ax.bar(x + offset, positives, 0.4, color=GREEN, alpha=alpha, label=f"{side} positives")
    ax.bar(x + offset, support[f"n_{side}"] - positives, 0.4, bottom=positives, color=GREY,
           alpha=alpha, label=f"{side} negatives")
ax.set_xticks(x, support.index)
ax.set_ylabel("rows")
ax.set_title("Rows per task: train side (left bar) and evaluation side (right bar)")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## A model never scores a user it trained on

Every downstream model in the series, the raw-feature baseline and the
fine-tuned TFM alike (notebook 04), fits on some users and is scored on
others. Each task divides its users once, with a fixed seed. For the fraud
tasks, the split is fixed before the splice, so a victim and its donor always land on the same side, and the
split refuses to build if any user would appear on both.

In [ ]:
for name, task in tasks.items():
    train_users = {s.entity_id for s in task.train_sequences}
    eval_users = {s.entity_id for s in task.eval_sequences}
    print(f"{name:>13}: {len(train_users):,} train users, {len(eval_users):,} evaluation users, "
          f"{len(train_users & eval_users)} shared")

## What the model may not see

The data slot is where leakage is prevented, so later slots can assume it:

- no event after a sequence's cutoff;
- no label, label-derived field, or post-outcome field as an input;
- profile milestones dated after the cutoff are dropped;
- the tokenizer is fitted on the train side only;
- spliced events never reach the pretraining corpus.


## Bringing your own data

Nothing downstream of this notebook knows the data is pragmatiq's. A book
reaches the model through an *adapter* that yields `TransactionEvent`s (user,
UTC timestamp, categorical, numeric, and text fields), `EntityProfile`s, and
optional label rows. To use another dataset, write an adapter for it and set
`data.adapter`. `tfm.contracts.schema` defines the records an adapter must yield.

### Exercises

1. Sample more users with `fetch_lab_assets("01", n_users=10_000)` (about 7.7 GB of host memory per 1,000 users) and rerun the task-support cell. Which task gains the most evaluation positives?
2. Pick a different evaluation-side victim from `splice["records"]` and redraw the theft-day plot. What gives the copied payments away for that victim?
3. The book carries more labels than the five tasks the series scores (`sorted(data.label_rows)`). Which one would make a good sixth task, and what would its leakage risks be?

## Where this lives in the kit

This notebook explains; the recipe runs. Every cell calls `src/tfm`, so the
same code runs in [`recipes/blueprint_v2`](../../recipes/blueprint_v2), and no
reusable logic lives here.

The spliced task is `tfm.data.tasks.identity_theft`; the asset download is
`tfm.runtime.lab`, which also holds the list of the other four tasks
(`LAB_TASKS`); `tfm.runtime.evaluate.fraud_task` builds `fraud_user`. The
checkpoint was pretrained on all 100,000 users.

Next: [02 · Event representation](02_event_representation.ipynb).